## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Cambia 'archive' por el nombre de la carpeta que subiste a Drive
    RUTA_DRIVE = '/content/drive/MyDrive/archive'
    os.chdir(RUTA_DRIVE)
    print(f"Directorio: {os.getcwd()}")
    # Instalar dependencias
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Entorno local — directorio: {os.getcwd()}")

Entorno local — directorio: c:\Users\mdura\Desktop\HRIA\HRIA


# EDA Proyecto I — Fase 3: Análisis Estadístico y Detección de Sesgos
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

En esta fase calculamos estadísticos descriptivos, exploramos correlaciones, analizamos grupos con `.groupby()` y — lo más crítico — **identificamos sesgos presentes en el dataset** con reflexión sobre su impacto en decisiones de negocio y modelos predictivos.

## 1. Carga de datos limpios

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Los CSV están en la misma carpeta del proyecto
df     = pd.read_csv('data_roles_completo.csv', low_memory=False)
df_sal = pd.read_csv('data_roles_salario.csv',  low_memory=False)

print(f"Dataset completo (roles de datos): {df.shape}")
print(f"Dataset con salario limpio:        {df_sal.shape}")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3275678103.py, line 8)

## 2. Estadística descriptiva — Variables clave

### 2.1 Salario anual (salary_annual)

In [ ]:
s = df_sal['salary_annual']

print("=== Salario Anual (USD) ===")
print(f"  Recuento           : {s.count():>12,}")
print(f"  Media              : ${s.mean():>12,.0f}")
print(f"  Mediana (P50)      : ${s.median():>12,.0f}")
print(f"  Desviación estándar: ${s.std():>12,.0f}")
print(f"  Mínimo             : ${s.min():>12,.0f}")
print(f"  Percentil 25 (Q1)  : ${s.quantile(0.25):>12,.0f}")
print(f"  Percentil 75 (Q3)  : ${s.quantile(0.75):>12,.0f}")
print(f"  Máximo             : ${s.max():>12,.0f}")
print(f"  Asimetría (skew)   :  {s.skew():>12.3f}")
print(f"  Curtosis           :  {s.kurtosis():>12.3f}")

**Interpretación:** La diferencia positiva entre media y mediana confirma una **distribución con cola derecha**: existen algunos salarios muy altos que elevan la media por encima del valor típico. La asimetría positiva (skew > 0) lo cuantifica. Para orientar candidatos, la **mediana es el indicador más representativo** del salario que esperaría una persona con ese rol, ya que no está distorsionado por los valores extremos.

### 2.2 Vistas de oferta (views)

In [ ]:
v = df['views']
print("=== Vistas por Oferta ===")
print(f"  Media              : {v.mean():>10,.1f}")
print(f"  Mediana (P50)      : {v.median():>10,.1f}")
print(f"  Desviación estándar: {v.std():>10,.1f}")
print(f"  P25                : {v.quantile(0.25):>10,.1f}")
print(f"  P75                : {v.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {v.max():>10,.1f}")

**Interpretación:** La alta desviación estándar respecto a la media indica gran variabilidad: algunas ofertas reciben miles de vistas mientras la mayoría pasa desapercibida. Esto refleja el **efecto de reputación de marca**: las grandes empresas tech concentran la atención.

### 2.3 Solicitudes recibidas (applies)

In [ ]:
a = df['applies']
print("=== Solicitudes por Oferta ===")
print(f"  Media              : {a.mean():>10,.1f}")
print(f"  Mediana (P50)      : {a.median():>10,.1f}")
print(f"  Desviación estándar: {a.std():>10,.1f}")
print(f"  P25                : {a.quantile(0.25):>10,.1f}")
print(f"  P75                : {a.quantile(0.75):>10,.1f}")
print(f"  Máximo             : {a.max():>10,.1f}")

**Interpretación:** La mediana de solicitudes es considerablemente más baja que la media, lo que confirma que pocas ofertas concentran la mayoría de las candidaturas (distribución de Pareto). Esto tiene implicaciones directas para DataTalent: los candidatos deben diferenciarse para destacar en las ofertas con mayor competencia.

## 3. Matriz de correlaciones (.corr())

In [ ]:
num_cols = [c for c in ['salary_annual','views','applies','employee_count','follower_count']
            if c in df_sal.columns]

corr = df_sal[num_cols].corr()
print("Matriz de correlaciones (Pearson):")
print(corr.round(3).to_string())

**Interpretación de la matriz de correlaciones:**
- **`views` ↔ `applies`:** correlación positiva esperada — más visibilidad genera más candidaturas.
- **`salary_annual` ↔ `employee_count`:** si positiva, las grandes empresas pagan más (economías de escala y presupuestos mayores).
- **`follower_count` ↔ `views`:** empresas con más seguidores en LinkedIn atraen más tráfico a sus ofertas.
- **`salary_annual` ↔ `views`:** correlación cercana a 0 indica que publicar el salario no determina significativamente la visibilidad de la oferta.

Nota: correlación no implica causalidad. Estos valores cuantifican relaciones lineales; relaciones no lineales pueden existir y no quedar capturadas aquí.

## 4. Análisis por grupos (.groupby() y pivot_table())

### 4.1 Salario por nivel de experiencia

In [ ]:
if 'formatted_experience_level' in df_sal.columns:
    exp_sal = (df_sal.groupby('formatted_experience_level')['salary_annual']
               .agg(Media='mean', Mediana='median', N_ofertas='count', Std='std')
               .round(0)
               .sort_values('Mediana', ascending=False))
    print("Salario anual por nivel de experiencia:")
    print(exp_sal.to_string())

**Interpretación:** La escalera salarial (Entry < Mid-Senior < Director < Executive) es una de las variables más importantes para el diseño del programa de reskilling: cuantifica exactamente cuánto valor económico aporta cada nivel de cualificación. Si la diferencia entre Entry y Mid-Senior es grande, el ROI del reskilling es alto y justifica la inversión.

### 4.2 Número de ofertas y salario por tipo de contrato

In [ ]:
wt_sal = (df_sal.groupby('formatted_work_type')['salary_annual']
           .agg(Media='mean', Mediana='median', N_ofertas='count')
           .round(0)
           .sort_values('N_ofertas', ascending=False))
print("Por tipo de contrato:")
print(wt_sal.to_string())

### 4.3 Top industrias por ofertas y salario

In [ ]:
if 'job_industries_list' in df_sal.columns:
    df_ind = df_sal[['job_id','salary_annual','job_industries_list']].dropna(subset=['job_industries_list']).copy()
    df_ind['industry'] = df_ind['job_industries_list'].str.split(', ')
    df_ind_exp = df_ind.explode('industry')

    ind_stats = (df_ind_exp.groupby('industry')['salary_annual']
                 .agg(Salario_medio='mean', N_ofertas='count')
                 .round(0)
                 .sort_values('N_ofertas', ascending=False)
                 .head(15))
    print("Top 15 industrias — roles de datos con salario:")
    print(ind_stats.to_string())

### 4.4 Top skills más demandadas

In [ ]:
if 'job_skills_list' in df.columns:
    df_sk = df[['job_id','job_skills_list']].dropna(subset=['job_skills_list']).copy()
    df_sk['skill'] = df_sk['job_skills_list'].str.split(', ')
    df_sk_exp = df_sk.explode('skill')
    skill_counts = df_sk_exp['skill'].value_counts().head(15)
    print("Top 15 skills más demandadas en roles de datos:")
    for sk, cnt in skill_counts.items():
        pct = cnt / len(df) * 100
        print(f"  {str(sk):<35} {cnt:>6,}  ({pct:.1f}%)")

**Interpretación:** Las skills más demandadas definen directamente el currículo del programa de reskilling. Una persona que domine el top 5 ya cubre la mayoría de las ofertas del mercado. Nótese que las 35 categorías del dataset son amplias (no específicas como "Python" o "SQL"), lo que significa que cada categoría engloba múltiples herramientas concretas.

### 4.5 Tabla pivot: mediana salarial por experiencia × tipo de contrato

In [ ]:
try:
    pivot = pd.pivot_table(
        df_sal,
        values='salary_annual',
        index='formatted_experience_level',
        columns='formatted_work_type',
        aggfunc='median',
        fill_value=0
    ).round(0)
    print("Tabla pivot — Mediana salarial (USD) por experiencia y tipo de contrato:")
    print(pivot.to_string())
except Exception as e:
    print(f"Nota: {e}")

## 5. Probabilidad condicional P(A|B)

In [ ]:
# P(nivel senior | rol Data Scientist) vs P(senior | Data Engineer) vs P(senior | global)
df_p = df.copy()
df_p['is_senior'] = df_p['formatted_experience_level'].str.contains(
    'senior|director|executive', case=False, na=False
)
df_p['is_ds'] = df_p['title'].str.contains('scientist|science', case=False, na=False)
df_p['is_de'] = df_p['title'].str.contains('data engineer', case=False, na=False)
df_p['is_da'] = df_p['title'].str.contains('data analyst', case=False, na=False)

p_s_ds  = df_p.loc[df_p['is_ds'], 'is_senior'].mean()
p_s_de  = df_p.loc[df_p['is_de'], 'is_senior'].mean()
p_s_da  = df_p.loc[df_p['is_da'], 'is_senior'].mean()
p_s_all = df_p['is_senior'].mean()

print("=== Probabilidad Condicional — P(Senior | Rol) ===")
print(f"  P(Senior | Data Scientist) = {p_s_ds:.3f}  ({p_s_ds*100:.1f}%)")
print(f"  P(Senior | Data Engineer)  = {p_s_de:.3f}  ({p_s_de*100:.1f}%)")
print(f"  P(Senior | Data Analyst)   = {p_s_da:.3f}  ({p_s_da*100:.1f}%)")
print(f"  P(Senior | cualquier rol)  = {p_s_all:.3f}  ({p_s_all*100:.1f}%)")

**Interpretación:** La probabilidad condicional P(Senior | Data Scientist) nos dice qué porcentaje de las ofertas de Data Scientist requieren nivel senior. Comparada con la probabilidad marginal P(Senior), revela si el rol demanda más experiencia que la media. Esto es información concreta para que DataTalent diseñe rutas de reskilling realistas: no tiene sentido orientar candidatos junior a posiciones que estadísticamente exigen senior.

## 6. Detección de Sesgos en el Dataset

Los sesgos en los datos pueden llevar a decisiones empresariales erróneas y a modelos de IA discriminatorios. Identificamos **4 sesgos** con su origen y su impacto potencial.

### Sesgo 1: MNAR en datos salariales (*Missing Not At Random*)
**Descripción:** Las columnas salariales tienen entre el 71% y el 95% de valores nulos. Los datos no faltan aleatoriamente: existe una razón sistemática para que falten.

**Origen:** Las empresas eligen estratégicamente si publicar el salario. Las startups y PYMEs con salarios menos competitivos suelen ocultarlos para no desalentar candidatos.


In [ ]:
total = len(df)
con_salario = df['salary_annual'].notna().sum()
sin_salario = total - con_salario

print("=== Análisis MNAR — Datos Salariales ===")
print(f"Con salario publicado:    {con_salario:>7,}  ({con_salario/total*100:.1f}%)")
print(f"Sin salario publicado:    {sin_salario:>7,}  ({sin_salario/total*100:.1f}%)")
print()
# ¿Las empresas más grandes publican el salario más a menudo?
if 'comp_size' in df.columns:
    print("Tasa de publicación de salario por tamaño de empresa:")
    res = df.groupby('comp_size')['salary_annual'].apply(
        lambda x: f"{x.notna().mean()*100:.0f}%  ({x.notna().sum():,} de {len(x):,})"
    )
    print(res.to_string())

**Impacto en un modelo predictivo:** Un modelo entrenado solo con los datos con salario aprendería los salarios de empresas que voluntariamente los publican (generalmente las más grandes y con mejor remuneración). Consecuencia: **sobreestimaría sistemáticamente** los salarios del mercado real. DataTalent podría orientar a candidatos con expectativas infladas, generando frustración en el proceso de selección.

### Sesgo 2: Sesgo geográfico — El dataset no representa el mercado español
**Descripción:** El dataset proviene de LinkedIn global, con predominancia de ofertas en EEUU. Si se usa para orientar candidatos en España, los hallazgos salariales y de skills no son directamente aplicables.

**Origen:** El scraping de LinkedIn se realizó principalmente en mercados anglosajones (mayor penetración de LinkedIn). Las ofertas españolas están subrepresentadas.

In [ ]:
if 'comp_country' in df.columns:
    country_counts = df['comp_country'].value_counts().head(12)
    total_con_pais = df['comp_country'].notna().sum()
    print(f"Distribución geográfica (de {total_con_pais:,} registros con país):\n")
    for country, cnt in country_counts.items():
        pct = cnt / total_con_pais * 100
        bar = '█' * int(pct / 2)
        print(f"  {str(country):<30} {cnt:>6,}  ({pct:5.1f}%)  {bar}")
    print(f"\nRegistros sin país: {df['comp_country'].isna().sum():,}")

**Impacto en un modelo predictivo:** Un modelo entrenado con este dataset y aplicado al mercado español **transladaría estándares salariales estadounidenses** (generalmente mucho más altos) a un contexto europeo, generando expectativas irreales. Las skills demandadas también pueden diferir: el mercado español tiene mayor presencia de banca, turismo y sector público.

**Medida de mitigación:** Complementar con datos de Infojobs, Tecnoempleo o SEPE para el contexto español.

### Sesgo 3: Sesgo de selección — Solo ofertas publicadas en LinkedIn
**Descripción:** LinkedIn es dominante para perfiles tech, pero no representa el mercado completo. Infojobs, Indeed, portales gubernamentales y ofertas directas (sin plataforma) quedan excluidos.

**Origen:** El dataset se extrajo de una única fuente, creando un sesgo estructural hacia empresas con marca empleadora activa en LinkedIn.

In [ ]:
print("=== Sesgo de Selección — Fuente única ===")
print("Plataforma: LinkedIn Job Postings (Kaggle)")
print()
print("Empresas sobrerepresentadas: grandes tecnológicas con marca empleadora fuerte")
print("Empresas subrepresentadas:  PYMEs, sector público, consultoras pequeñas")
print()
if 'comp_size' in df.columns:
    print("Distribución de ofertas por tamaño de empresa:")
    cs = df['comp_size'].value_counts(dropna=False)
    for size, cnt in cs.items():
        pct = cnt / len(df) * 100
        print(f"  {str(size):<15} {cnt:>7,}  ({pct:.1f}%)")

**Impacto en un modelo predictivo:** El modelo aprendería que "una oferta de datos típica proviene de una gran empresa tech con perfil LinkedIn activo". Fallaría en predecir salarios y skills para roles en PYMEs o sector público, que representan una parte significativa del tejido empresarial español (especialmente relevante para candidatos de ciudades no metropolitanas).

### Sesgo 4: Ausencia de datos de género (*Undisclosed protected attribute*)
**Descripción:** El dataset no contiene información de género. Sin este dato, es imposible detectar directamente brechas salariales de género, uno de los análisis más críticos para garantizar equidad en el reskilling.

**Origen:** LinkedIn no incluye género por defecto en sus datos de ofertas de empleo. Incorporarlo requeriría autodeclaración o inferencia algorítmica, ambas con problemas legales y éticos bajo el RGPD europeo.

In [ ]:
print("=== Sesgo por Atributo Protegido Ausente (Género) ===")
print()
print("Variables de género: NO disponibles en el dataset")
print()
print("Consecuencias directas:")
print("  1. No podemos cuantificar la brecha salarial de género")
print("  2. Un modelo entrenado sin control de género puede perpetuar sesgos históricos")
print("  3. Las decisiones de reskilling sin perspectiva de género pueden ser excluyentes")
print()
# Proxy: distribución de roles que históricamente tienen sesgo de género
if 'title' in df.columns:
    tech_roles = df['title'].str.contains('engineer|developer|architect', case=False, na=False).sum()
    soft_roles = df['title'].str.contains('analyst|coordinator|manager', case=False, na=False).sum()
    print(f"Roles de ingeniería/desarrollo (históricamente masculinizados): {tech_roles:,} ({tech_roles/len(df)*100:.1f}%)")
    print(f"Roles analíticos/gestión (mayor diversidad de género): {soft_roles:,}  ({soft_roles/len(df)*100:.1f}%)")
    print()
    print("Nota: esta es una aproximación indirecta, no un análisis de género real.")

**Impacto en un modelo predictivo:** Un modelo de predicción salarial sin control de género puede aprender patrones discriminatorios implícitos: si históricamente las mujeres están subrepresentadas en roles senior de ingeniería de datos, el modelo aprenderá a predecir salarios más bajos para ciertos patrones de carrera. Esto infringiría el principio de equidad algorítmica (Fairness AI) y podría vulnerar legislación anti-discriminación europea.

**Medida de mitigación:** Cruzar con encuestas que incluyan autodeclaración de género (Stack Overflow Developer Survey, Kaggle ML Survey) para análisis de equidad de género.

## 7. Resumen del análisis estadístico

| Hallazgo | Métrica clave | Implicación para DataTalent |
|----------|-------------|----------------------------|
| Salario mediano en roles de datos | Ver análisis § 2.1 | Referencia para orientación salarial realista |
| Skills más demandadas | Top 5 del ranking | Define el currículo mínimo del programa de reskilling |
| ROI del nivel de experiencia | Diferencia median Entry vs Senior | Argumento económico para justificar la inversión en formación |
| Industrias con más demanda | Top 3 de § 4.3 | Sectores objetivo para colocación de candidatos |
| **MNAR salarial** | 71–95% de nulos | Las cifras salariales son una estimación parcial sesgada hacia arriba |
| **Sesgo geográfico** | Mayoría EEUU | Ajustar o complementar con datos españoles antes de usar |
| **Sin datos de género** | 0% cobertura | Requiere fuente externa para garantizar equidad en el programa |

**Próximo paso (Fase 4):** Visualización de todos estos hallazgos.